# Getting started: Australian land parcels + G-NAF

Every land parcel from the open state cadastres (NSW, VIC, QLD, TAS, ACT, WA), one file per state, each parcel
carrying the ids of its G-NAF addresses. This notebook loads the data, compares states, plots parcel polygons and
joins addresses.

The dataset is built by the open-source pipeline at
[github.com/nephorion/kaggle-au-cadastre](https://github.com/nephorion/kaggle-au-cadastre).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

# Locate the dataset wherever Kaggle mounts it.
DATA = os.path.dirname(next(iter(glob.glob("/kaggle/input/**/parquet", recursive=True))))
GNAF = next(iter(glob.glob("/kaggle/input/**/gnaf_csv", recursive=True)), None)   # optional second input
print("parcels:", DATA)
print("G-NAF:", GNAF)
sorted(os.listdir(f"{DATA}/parquet"))

## Load one state without geometry

Parquet reads only the columns you ask for. Leave out `wkt` unless you need the polygons.

In [ ]:
cols = ["state", "source_oid", "gnaf_legal_parcel_id", "gnaf_key_matched", "gnaf_address_count",
        "gnaf_address_detail_pids", "lot", "plan", "tenure", "area_m2", "locality", "point_lon", "point_lat"]
act = pd.read_parquet(f"{DATA}/parquet/ACT_parcels.parquet", columns=cols)
print(f"{len(act):,} ACT parcels")
act.head()

## All states in one table

The identity, G-NAF and harmonised columns are the same in every file, so states stack directly.

In [ ]:
common = ["state", "gnaf_key_matched", "gnaf_address_count", "status", "tenure", "is_strata", "area_m2"]
files = sorted(glob.glob(f"{DATA}/parquet/*_parcels.parquet"))
australia = pd.concat([pd.read_parquet(f, columns=common) for f in files], ignore_index=True)

by_state = australia.groupby("state").agg(
    parcels=("state", "size"),
    with_address=("gnaf_address_count", lambda s: (s > 0).mean()),
    key_matched=("gnaf_key_matched", "mean"),
    median_area_m2=("area_m2", "median"),
)
by_state.style.format({"parcels": "{:,}", "with_address": "{:.1%}", "key_matched": "{:.1%}",
                       "median_area_m2": "{:,.0f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(australia["gnaf_address_count"].clip(upper=10).value_counts().sort_index()
    .plot.bar(ax=ax, color="#2a9d8f", title="Addresses per parcel, all states (10 = 10 or more)"))
ax.set_xlabel("G-NAF addresses on the parcel")
ax.set_ylabel("parcels")
plt.show()

## Plot parcel polygons

`wkt` holds each parcel's boundary in EPSG:4326. Here are the parcels of one Canberra suburb, coloured by the
number of addresses on each.

In [ ]:
import geopandas as gpd

act_geo = pd.read_parquet(f"{DATA}/parquet/ACT_parcels.parquet",
                          columns=["source_oid", "locality", "gnaf_address_count", "wkt"])
suburb = act_geo[act_geo["locality"] == "BRADDON"]
gdf = gpd.GeoDataFrame(suburb, geometry=gpd.GeoSeries.from_wkt(suburb["wkt"]), crs="EPSG:4326")

ax = gdf.plot(column=gdf["gnaf_address_count"].clip(upper=20), cmap="viridis", legend=True,
              linewidth=0.2, edgecolor="white", figsize=(8, 8))
ax.set_title(f"Braddon, ACT: {len(gdf):,} parcels by address count")
ax.set_axis_off()
plt.show()

## Join G-NAF addresses

`gnaf_address_detail_pids` is a list per parcel: explode it to one row per address and merge with a flat copy of
G-NAF. This cell runs when a flat G-NAF dataset (files `gnaf_csv/gnaf_<STATE>_*.csv`) is added as a second input.

In [ ]:
if GNAF:
    gnaf_cols = ["address_detail_pid", "full_address", "locality_name", "postcode_effective"]
    gnaf = pd.concat([pd.read_csv(f, usecols=gnaf_cols, dtype=str)
                      for f in glob.glob(f"{GNAF}/gnaf_ACT_*.csv")], ignore_index=True)
    parcel_addresses = (act.explode("gnaf_address_detail_pids")
                           .merge(gnaf, how="left", left_on="gnaf_address_detail_pids",
                                  right_on="address_detail_pid"))
    display(parcel_addresses[["source_oid", "gnaf_legal_parcel_id", "full_address", "locality_name"]]
            .dropna(subset=["full_address"]).head(10))
else:
    print("Add a flat G-NAF dataset as a second input to run the join.")